In [1]:
import uuid

import httpx
from a2a.client import A2ACardResolver, create_client
from a2a.types import (
    AgentCard,
    Message,
    Part,
    Role,
    SendMessageRequest,
)
from google.protobuf.json_format import MessageToJson

In [3]:
BASE_URL = "http://localhost:10001"
PUBLIC_AGENT_CARD_PATH = "/.well-known/agent-card.json"


In [4]:
async with httpx.AsyncClient() as httpx_client:

    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=BASE_URL,
    )

    try:
        print(
            f"Fetching public agent card from: "
            f"{BASE_URL}{PUBLIC_AGENT_CARD_PATH}"
        )

        public_card = await resolver.get_agent_card()

        print("Fetched public agent card")
        print(MessageToJson(public_card, indent=2))

    except Exception as e:
        print(f"Error fetching public agent card: {e}")

Fetching public agent card from: http://localhost:10001/.well-known/agent-card.json
Fetched public agent card
{
  "name": "Warehouse Manager Agent",
  "description": "The user is asking to reserve items from the warehouses or about availability of items in the warehouses.",
  "supportedInterfaces": [
    {
      "url": "http://localhost:10001/",
      "protocolBinding": "JSON-RPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "skills": [
    {
      "id": "ABC",
      "name": "Check Warehouse Availability",
      "description": "Check availability of items across warehouses.",
      "tags": [
        "availability",
        "warehouse",
        "warehouses"
      ],
      "examples": [
        "What is the availability of the item 123?"
      ]
    },
    {
      "id": "DEF",
      "name": "Reserve Warehouse Items",
      "description": 

In [8]:
timeout_config = httpx.Timeout(
    connect=10.0,  # Connection timeout
    read=100.0,    # Read timeout (important for long-running operations)
    write=10.0,    # Write timeout
    pool=10.0,     # Pool timeout
)

client = await create_client(
    agent=BASE_URL,
    resolver_http_kwargs={"timeout": timeout_config},
)

print("A2AClient initialized")

message_id = str(uuid.uuid4())
message_payload = Message(
    role=Role.ROLE_USER,
    message_id=message_id,
    parts=[Part(text="Hello, how are you?")],
)

request = SendMessageRequest(message=message_payload)

async for response in client.send_message(request):
    print(response)

A2AClient initialized
task {
  id: "a647ade7-fef1-4911-b1d0-769f8af92881"
  context_id: "daab94aa-8f10-4936-90b9-2d397db55a1d"
  status {
    state: TASK_STATE_SUBMITTED
  }
}

status_update {
  task_id: "a647ade7-fef1-4911-b1d0-769f8af92881"
  context_id: "daab94aa-8f10-4936-90b9-2d397db55a1d"
  status {
    state: TASK_STATE_WORKING
    timestamp {
      seconds: 1786517541
      nanos: 92629000
    }
  }
}

artifact_update {
  task_id: "a647ade7-fef1-4911-b1d0-769f8af92881"
  context_id: "daab94aa-8f10-4936-90b9-2d397db55a1d"
  artifact {
    artifact_id: "b44c8a5c-e438-4779-a4a0-5b8a5c023093"
    name: "response"
    parts {
      text: "Actions performed:\n- No warehouse actions were performed.\n\nResponse:\n- Hello! I’m doing well, thanks for asking. How can I help you today?"
    }
  }
}

status_update {
  task_id: "a647ade7-fef1-4911-b1d0-769f8af92881"
  context_id: "daab94aa-8f10-4936-90b9-2d397db55a1d"
  status {
    state: TASK_STATE_COMPLETED
    timestamp {
      seconds: 